# MMA 869 Project: Example Notebook

*Updated Jan 16, 2026*

This notebook serves as a template for the Team Project. Teams can use this notebook as a starting point, and update it successively with new ideas and techniques to improve their model results.

Note that is not required to use this template. Teams may also alter this template in any way they see fit.

# Preliminaries: Inspect and Set up environment

No action is required on your part in this section. These cells print out helpful information about the environment, just in case.

In [48]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report

In [49]:
# TODO: if you need to install any package, do so here. For example:
#pip install unidecode

In [50]:
!pip install catboost

# 0: Data Loading and Inspection

In [51]:
df = pd.read_csv("https://raw.githubusercontent.com/stepthom/869_course/refs/heads/main/data/resort_train.csv")

In [52]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6954 entries, 0 to 6953
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   GuestID         6954 non-null   int64  
 1   BookingDate     6954 non-null   object 
 2   PromoCode       3709 non-null   object 
 3   Region          6785 non-null   object 
 4   AllInclusive    6786 non-null   float64
 5   Room            6568 non-null   object 
 6   PackageType     6801 non-null   object 
 7   Age             6478 non-null   float64
 8   VIP             6796 non-null   float64
 9   RoomService     6490 non-null   float64
 10  Dining          6466 non-null   float64
 11  Retail          6790 non-null   float64
 12  Spa             6806 non-null   float64
 13  Entertainment   6815 non-null   float64
 14  LoyaltyPoints   6954 non-null   int64  
 15  SurveyScore     6954 non-null   int64  
 16  DaysSinceEmail  6954 non-null   int64  
 17  BookingChannel  6954 non-null   o

In [53]:
# Let's print some descriptive statistics for all the numeric features.

df.describe().T

,count,mean,std,min,25%,50%,75%,max
GuestID,6954.0,544815.121369,258449.781823,100000.0,322043.25,542402.5,769552.75,999943.000000
AllInclusive,6786.0,0.358974,0.479735,0.0,0.00,0.0,1.00,1.000000
Age,6478.0,28.682001,14.530100,0.0,19.00,27.0,38.00,79.000000
VIP,6796.0,0.023396,0.151169,0.0,0.00,0.0,0.00,1.000000
RoomService,6490.0,3823.687190,26416.086096,0.0,0.00,0.0,84.00,282324.947439
Dining,6466.0,7651.359101,54005.989936,0.0,0.00,0.0,95.00,590932.008676
Retail,6790.0,5980.799434,43130.400968,0.0,0.00,0.0,37.75,468882.332928
Spa,6806.0,5728.859525,41105.729649,0.0,0.00,0.0,80.00,447858.846683
Entertainment,6815.0,6757.871818,46162.309423,0.0,0.00,0.0,67.00,482032.266355
LoyaltyPoints,6954.0,5037.915876,2870.166683,0.0,2566.25,5055.5,7496.75,9996.000000


In [54]:
# Let's print some descriptive statistics for all the numeric features.

df.describe().T# What is the number of unique values in all the categorical features? And what is
# the value with the highest frequency?

df.describe(include=object).T

,count,unique,top,freq
BookingDate,6954,658,2024-01-20,59
PromoCode,3709,2,PromoB,1884
Region,6785,3,Americas,3673
Room,6568,5297,E/13/S,7
PackageType,6801,3,Relaxation,4694
BookingChannel,6954,5,Phone,1438
AgeGroup,6671,5,Young,2586
ReferralSource,6954,19,Google,1235


In [55]:
# How much missing data is in each feature?

df.isna().sum()

,0
GuestID,0
BookingDate,0
PromoCode,3245
Region,169
AllInclusive,168
Room,386
PackageType,153
Age,476
VIP,158
RoomService,464


In [56]:
# For convienience, let's save the names of all numeric features to a list,
# and the names of all categorical features to another list.

numeric_features = df.select_dtypes(include='number').columns.tolist()
print(numeric_features)

categorical_features = df.select_dtypes(include='object').columns.tolist()
print(categorical_features)

['GuestID', 'AllInclusive', 'Age', 'VIP', 'RoomService', 'Dining', 'Retail', 'Spa', 'Entertainment', 'LoyaltyPoints', 'SurveyScore', 'DaysSinceEmail', 'Churned']
['BookingDate', 'PromoCode', 'Region', 'Room', 'PackageType', 'BookingChannel', 'AgeGroup', 'ReferralSource']


In [57]:
# For convienience, let's save the names of all numeric features to a list,
# and the names of all categorical features to another list.

numeric_features = df.select_dtypes(include='number').columns.tolist()
print(numeric_features)

categorical_features = df.select_dtypes(include='object').columns.tolist()
print(categorical_features)

['GuestID', 'AllInclusive', 'Age', 'VIP', 'RoomService', 'Dining', 'Retail', 'Spa', 'Entertainment', 'LoyaltyPoints', 'SurveyScore', 'DaysSinceEmail', 'Churned']
['BookingDate', 'PromoCode', 'Region', 'Room', 'PackageType', 'BookingChannel', 'AgeGroup', 'ReferralSource']


In [58]:
# TODO: Can add more EDA here, as desired

# 1: Pipeline 1: Simple Feature Engineering and then Decision Tree

In [59]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split

In [60]:
# Scikit-learn needs us to put the features in one dataframe, and the label in another.
# It's tradition to name these variables X and y, but it doesn't really matter.

X = df.drop(['GuestID', 'BookingDate','Churned'], axis=1)
y = df['Churned']
df.head()

,GuestID,BookingDate,PromoCode,Region,AllInclusive,Room,PackageType,Age,VIP,RoomService,...,Retail,Spa,Entertainment,LoyaltyPoints,SurveyScore,DaysSinceEmail,BookingChannel,AgeGroup,ReferralSource,Churned
0,619623,2024-02-10,NaN,Americas,0.0,G/630/S,Relaxation,0.0,0.0,0.0,...,0.0,0.0,0.0,6915,5,136,Website,NaN,Facebook,1
1,776250,2024-01-03,NaN,Americas,1.0,G/201/S,Relaxation,17.0,0.0,0.0,...,0.0,0.0,0.0,8571,5,362,Corporate,Minor,Billboard,1
2,932709,2023-01-17,NaN,Americas,NaN,G/1483/S,Wellness,35.0,0.0,0.0,...,0.0,0.0,0.0,1142,4,154,TravelAgent,Middle,Facebook,0
3,771839,2023-12-09,PromoA,Europe,1.0,D/164/S,Adventure,26.0,0.0,0.0,...,0.0,NaN,0.0,9642,2,128,Website,Young,Magazine,1
4,755501,2024-02-15,PromoA,Americas,0.0,G/818/P,Relaxation,13.0,0.0,0.0,...,60.0,1.0,5147.0,5528,4,35,Mobile,Minor,Google,0


In [61]:
def remove_duplicates(df, name="DataFrame"):
    initial_rows = df.shape[0]
    df.drop_duplicates(inplace=True)
    final_rows = df.shape[0]
    if initial_rows != final_rows:
        print(f"Removed {initial_rows - final_rows} duplicate rows from {name}.")
    return df

## 1.1: Cleaning and FE

In [62]:
# Initial DataFrame preparation: Drop irrelevant columns and handle PromoCode NaNs
X = df.drop(['GuestID', 'BookingDate','Churned'], axis=1)

encoded_features = ['ReferralSource', 'BookingChannel', 'AgeGroup', 'PackageType', 'Region', 'PromoCode']

# Fill NaN values in 'PromoCode' with 'NA' so it can be encoded as a category
X['PromoCode'] = X['PromoCode'].fillna('NA')

# Fill NaN values in 'Entertainment' with 0
X['Entertainment'] = X['Entertainment'].fillna(0)

# Fill NaN values in 'Dining' with 0
X['Dining'] = X['Dining'].fillna(0)

# Fill NaN values in 'RoomService' with 0
X['RoomService'] = X['RoomService'].fillna(0)

# Fill NaN values in 'Retail' with 0
X['Retail'] = X['Retail'].fillna(0)

# Fill NaN values in 'Spa' with 0
X['Spa'] = X['Spa'].fillna(0)

# Calculate the mean of 'Entertainment' for values less than or equal to 20336
mean_entertainment_for_replacement = X[X['Entertainment'] <= 20336]['Entertainment'].mean()
print(f"Mean of Entertainment column (values <= 20336): {mean_entertainment_for_replacement}")

# Replace values in 'Entertainment' greater than 20336 with the calculated mean
X.loc[X['Entertainment'] > 20336, 'Entertainment'] = mean_entertainment_for_replacement

# Calculate the mean of 'Dining' for values less than or equal to 29813
mean_dining_for_replacement = X[X['Dining'] <= 29813]['Dining'].mean()
print(f"Mean of Dining column (values <= 29813): {mean_dining_for_replacement}")

# Replace values in 'Dining' greater than 29813 with the calculated mean
X.loc[X['Dining'] > 29813, 'Dining'] = mean_dining_for_replacement

# Calculate the mean of 'RoomService' for values less than or equal to 14327
mean_roomservice_for_replacement = X[X['RoomService'] <= 14327]['RoomService'].mean()
print(f"Mean of RoomService column (values <= 14327): {mean_roomservice_for_replacement}")

# Replace values in 'RoomService' greater than 14327 with the calculated mean
X.loc[X['RoomService'] > 14327, 'RoomService'] = mean_roomservice_for_replacement

# Calculate the mean of 'Retail' for values less than or equal to 12253
mean_retail_for_replacement = X[X['Retail'] <= 12253]['Retail'].mean()
print(f"Mean of Retail column (values <= 12253): {mean_retail_for_replacement}")

# Replace values in 'Retail' greater than 12253 with the calculated mean
X.loc[X['Retail'] > 12253, 'Retail'] = mean_retail_for_replacement

# Calculate the mean of 'Spa' for values less than or equal to 18572
mean_spa_for_replacement = X[X['Spa'] <= 18572]['Spa'].mean()
print(f"Mean of Spa column (values <= 18572): {mean_spa_for_replacement}")

# Replace values in 'Spa' greater than 18572 with the calculated mean
X.loc[X['Spa'] > 18572, 'Spa'] = mean_spa_for_replacement

# Apply one-hot encoding to the specified categorical features
X = pd.get_dummies(X, columns=encoded_features, dummy_na=False)

# Convert boolean columns (from get_dummies) to integers
for col in X.select_dtypes(include='bool').columns:
    X[col] = X[col].astype(int)

# Drop any remaining categorical features that were not specified for encoding
# (e.g., 'BookingDate' and 'Room' from the original categorical_features list, which may have too many unique values for one-hot encoding)
# We'll drop 'BookingDate' as it often requires specific time-series handling, and 'Room' due to its high cardinality.
# The 'errors='ignore'' argument prevents an error if the column is already gone (e.g., after get_dummies).
X = X.drop(['BookingDate', 'Room'], axis=1, errors='ignore')

# Store column names before imputation, as X will become a NumPy array
X_columns_after_preprocessing = X.columns.tolist()

# Impute missing numerical values after all columns are numeric
from sklearn.impute import SimpleImputer # Ensure SimpleImputer is imported if not globally
imp = SimpleImputer(strategy='constant', fill_value=0) # Instantiate SimpleImputer with constant strategy and fill_value 0
X = imp.fit_transform(X).astype(int) # Fit and transform X

display(pd.DataFrame(X, columns=X_columns_after_preprocessing).head())

Mean of Entertainment column (values <= 20336): 293.3220214485089
Mean of Dining column (values <= 29813): 419.34153846153845
Mean of RoomService column (values <= 14327): 217.93975373790678
Mean of Retail column (values <= 12253): 166.40950425344676
Mean of Spa column (values <= 18572): 298.87483511651766


,AllInclusive,Age,VIP,RoomService,Dining,Retail,Spa,Entertainment,LoyaltyPoints,SurveyScore,...,AgeGroup_Young,PackageType_Adventure,PackageType_Relaxation,PackageType_Wellness,Region_Americas,Region_AsiaPacific,Region_Europe,PromoCode_NA,PromoCode_PromoA,PromoCode_PromoB
0,0,0,0,0,0,0,0,0,6915,5,...,0,0,1,0,1,0,0,1,0,0
1,1,17,0,0,0,0,0,0,8571,5,...,0,0,1,0,1,0,0,1,0,0
2,0,35,0,0,419,0,0,0,1142,4,...,0,0,0,1,1,0,0,1,0,0
3,1,26,0,0,0,0,0,0,9642,2,...,1,1,0,0,0,0,1,0,1,0
4,0,13,0,0,0,60,1,5147,5528,4,...,0,0,1,0,1,0,0,0,1,0


In [63]:
X_processed_df = pd.DataFrame(X, columns=X_columns_after_preprocessing)
X_processed_df.to_excel('X_processed_data.xlsx', index=False)
print("Preprocessed and encoded data saved to 'X_processed_data.xlsx'")

Preprocessed and encoded data saved to 'X_processed_data.xlsx'


The file `X_processed_data.xlsx` has been saved in your Colab environment. You can download it from the file browser on the left panel to inspect the encoded data.

## 1.3: Data Splitting

Before training multiple models, we'll split our preprocessed data into training and testing sets. This ensures that we evaluate our models on unseen data, providing a more reliable estimate of their performance.

In [64]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (5563, 49)
X_test shape: (1391, 49)
y_train shape: (5563,)
y_test shape: (1391,)


## 1.3.1: Feature Scaling

To improve the performance and convergence of some models (especially Logistic Regression), it's beneficial to scale the features. We'll use `StandardScaler` to transform the data such that it has a mean of 0 and a standard deviation of 1. It's crucial to fit the scaler only on the training data to avoid data leakage.

In [65]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform both training and test data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data scaled successfully.")


Data scaled successfully.


## 1.4: Evaluating Multiple Algorithms

Now, let's set up a loop to train and evaluate different classification algorithms using the F1 macro score. We will include:
- Random Forest
- Gradient Boosting
- Extra Trees
- Logistic Regression
- CatBoost Classifier

In [89]:
from sklearn.ensemble import GradientBoostingClassifier, ExtraTreesClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import xgboost as xgb


from sklearn.metrics import f1_score

models = {
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42,
                                            min_samples_split=10, n_estimators=500,
                                            max_samples=0.7),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42 ),
    "Extra Trees": ExtraTreesClassifier(class_weight='balanced', random_state=42),
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=5000, random_state=42),
    "CatBoost Classifier": CatBoostClassifier(iterations=1000,
                            depth=4,
                            learning_rate=0.2255611979301099,
                            l2_leaf_reg=0.2939990463887897,
                            bagging_temperature=0.13866256060046356,
                            random_strength=0.02945655242478888,
                            border_count=212,
                            min_data_in_leaf=52,
                           # scale_pos_weight=neg / pos,
                            eval_metric="AUC",
                            early_stopping_rounds=50,
                            random_seed=20,
                            verbose=False),
    "xgboost": xgb.XGBClassifier(random_state=42, n_estimators=25, max_depth=8, iterations=1000 ,learning_rate=0.1, early_stopping=400),
    "LightGBM": LGBMClassifier(random_state=42, n_estimators=100, max_depth=8, max_bin=64, learning_rate=0.01)
    # Add more models as needed
    # Add more models as needed

}

results = {}

for name, m in models.items():
    print(f"\nTraining and evaluating {name}...")
    # Use scaled data for training and evaluation
    m.fit(X_train_scaled, y_train)
    pred = m.predict(X_test_scaled)
    score = f1_score(y_test, pred, average='macro')
    results[name] = score
    print(f"{name} - F1 Macro: {score:.10f}")

print("\n--- Model Evaluation Summary ---")
for name, score in results.items():
    print(f"{name}: {score:.10f}")


Training and evaluating Random Forest...
Random Forest - F1 Macro: 0.8114489646

Training and evaluating Gradient Boosting...
Gradient Boosting - F1 Macro: 0.8130832427

Training and evaluating Extra Trees...
Extra Trees - F1 Macro: 0.7758842475

Training and evaluating Logistic Regression...
Logistic Regression - F1 Macro: 0.8123648170

Training and evaluating CatBoost Classifier...
CatBoost Classifier - F1 Macro: 0.7979452887

Training and evaluating xgboost...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [19:32:04] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "early_stopping", "iterations" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


xgboost - F1 Macro: 0.8115402277

Training and evaluating LightGBM...
[LightGBM] [Info] Number of positive: 2801, number of negative: 2762
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003538 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 638
[LightGBM] [Info] Number of data points in the train set: 5563, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503505 -> initscore=0.014021
[LightGBM] [Info] Start training from score 0.014021
LightGBM - F1 Macro: 0.7888555773

--- Model Evaluation Summary ---
Random Forest: 0.8114489646
Gradient Boosting: 0.8130832427
Extra Trees: 0.7758842475
Logistic Regression: 0.8123648170
CatBoost Classifier: 0.7979452887
xgboost: 0.8115402277
LightGBM: 0.7888555773


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### Detailed Classification Reports

Let's get a more detailed view of each model's performance by printing the `classification_report` for each.

In [84]:
from sklearn.metrics import classification_report

for name, m in models.items():
    print(f"\n--- Classification Report for {name} ---")
    pred = m.predict(X_test_scaled)
    print(classification_report(y_test, pred))



--- Classification Report for Random Forest ---
              precision    recall  f1-score   support

           0       0.79      0.85      0.82       690
           1       0.84      0.77      0.81       701

    accuracy                           0.81      1391
   macro avg       0.81      0.81      0.81      1391
weighted avg       0.81      0.81      0.81      1391


--- Classification Report for Gradient Boosting ---
              precision    recall  f1-score   support

           0       0.81      0.82      0.81       690
           1       0.82      0.81      0.81       701

    accuracy                           0.81      1391
   macro avg       0.81      0.81      0.81      1391
weighted avg       0.81      0.81      0.81      1391


--- Classification Report for Extra Trees ---
              precision    recall  f1-score   support

           0       0.75      0.83      0.79       690
           1       0.81      0.72      0.76       701

    accuracy                     

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [68]:
# We know this dataset has some missing data, and we also know that DTs don't
# allow missing data. For now, we'll just do simple imputation.
#
# TODO: consider doing something different/better, like impute them (as
# discussed in class)

# Imputation logic has been moved to cell Sl4aB8vMnHAW to ensure it's applied after
# categorical features are properly encoded and converted to numeric types.

In [69]:
# We know this dataset has some missing data, and we also know that DTs don't
# allow missing data. For now, we'll just do simple imputation.
#
# TODO: consider doing something different/better, like impute them (as
# discussed in class)

# Imputation logic has been moved to cell Sl4aB8vMnHAW to ensure it's applied after
# categorical features are properly encoded and converted to numeric types.

In [70]:
# TODO: Add more data cleaning and FE, as desired.

## 1.2: Model creation, hyperparameter tuning, and validation

In [71]:
# Let's create a very simple DecisionTree.

dtModel = DecisionTreeClassifier(max_depth=8, random_state=20, criterion='gini',
                                 min_samples_split=2, class_weight='balanced', splitter='best'
                                 )
#

# TODO: Can try different algorithms

In [72]:
# We use cross_validate to perform K-fold cross validation for us.
cv_results = cross_validate(dtModel, X, y, cv=5, scoring="f1_macro")

# TODO: can also add hyperparameter tuning to explore different values of the algorithms
# hyperparameters, and see how much those affect results.
# See GridSearchCV or RandomizedSearchCV.

In [73]:
# Now that cross validation has completed, we can see what it estimates the peformance
# of our model to be.
#min_samples_leaf=7 increased by 0.0009
display(cv_results)
print("The mean CV score is:")
print(np.mean(cv_results['test_score']))

{'fit_time': array([0.10150933, 0.084589  , 0.12528324, 0.15405488, 0.10617948]),
 'score_time': array([0.00572848, 0.00569582, 0.0105207 , 0.00740981, 0.01122212]),
 'test_score': array([0.78628971, 0.78993045, 0.78099349, 0.77950282, 0.77548333])}

The mean CV score is:
0.782439956488265



Once we are happy with the estimated performance of our model, we can move on to the final step.

First, we train our model one last time, using all available training data (unlike CV, which always uses a subset). This final training will give our model the best chance as the highest performance.

Then, we must load in the (unlabeled) competition data from the cloud and use our model to generate predictions for each instance in that data. We will then output those predictions to a CSV file and upload it to the competition.

In [74]:
# Our model's "final form"
bkp_X = X
clf = dtModel.fit(bkp_X, y)

print(f"The depth of the decision tree is: {clf.get_depth()}")
print(f"To get the max leaf nodes of the decision tree is: {clf.get_n_leaves()}")

The depth of the decision tree is: 8
To get the max leaf nodes of the decision tree is: 170


In [75]:
X_comp = pd.read_csv("https://raw.githubusercontent.com/stepthom/869_course/refs/heads/main/data/resort_test.csv")

# Will need to save these IDs for later
passengerIDs = X_comp["GuestID"]

# Remove duplicate records from the competition data
X_comp = remove_duplicates(X_comp, name="Competition DataFrame")

# Importantly, we need to perform the same cleaning/transformation steps
# on this competition data as you did the training data. Otherwise, we will
# get an error and/or unexpected results.

X_comp = X_comp.drop(['GuestID'], axis=1, errors='ignore')

# Fill NaN values in 'PromoCode' with 'NA' for competition data
X_comp['PromoCode'] = X_comp['PromoCode'].fillna('NA')

# Fill NaN values in 'Entertainment' with 0 for competition data
X_comp['Entertainment'] = X_comp['Entertainment'].fillna(0)

# Fill NaN values in 'Dining' with 0 for competition data
X_comp['Dining'] = X_comp['Dining'].fillna(0)

# Fill NaN values in 'RoomService' with 0 for competition data
X_comp['RoomService'] = X_comp['RoomService'].fillna(0)

# Fill NaN values in 'Retail' with 0 for competition data
X_comp['Retail'] = X_comp['Retail'].fillna(0)

# Fill NaN values in 'Spa' with 0 for competition data
X_comp['Spa'] = X_comp['Spa'].fillna(0)

# Calculate the mean of 'Entertainment' for values less than or equal to 20336 for competition data
mean_entertainment_for_replacement_comp = X_comp[X_comp['Entertainment'] <= 20336]['Entertainment'].mean()

# Replace values in 'Entertainment' greater than 20336 with the calculated mean for competition data
X_comp.loc[X_comp['Entertainment'] > 20336, 'Entertainment'] = mean_entertainment_for_replacement_comp

# Calculate the mean of 'Dining' for values less than or equal to 29813 for competition data
mean_dining_for_replacement_comp = X_comp[X_comp['Dining'] <= 29813]['Dining'].mean()

# Replace values in 'Dining' greater than 29813 with the calculated mean for competition data
X_comp.loc[X_comp['Dining'] > 29813, 'Dining'] = mean_dining_for_replacement_comp

# Calculate the mean of 'RoomService' for values less than or equal to 14327 for competition data
mean_roomservice_for_replacement_comp = X_comp[X_comp['RoomService'] <= 14327]['RoomService'].mean()

# Replace values in 'RoomService' greater than 14327 with the calculated mean for competition data
X_comp.loc[X_comp['RoomService'] > 14327, 'RoomService'] = mean_roomservice_for_replacement_comp

# Calculate the mean of 'Retail' for values less than or equal to 12253 for competition data
mean_retail_for_replacement_comp = X_comp[X_comp['Retail'] <= 12253]['Retail'].mean()

# Replace values in 'Retail' greater than 12253 with the calculated mean for competition data
X_comp.loc[X_comp['Retail'] > 12253, 'Retail'] = mean_retail_for_replacement_comp

# Calculate the mean of 'Spa' for values less than or equal to 18572 for competition data
mean_spa_for_replacement_comp = X_comp[X_comp['Spa'] <= 18572]['Spa'].mean()

# Replace values in 'Spa' greater than 18572 with the calculated mean for competition data
X_comp.loc[X_comp['Spa'] > 18572, 'Spa'] = mean_spa_for_replacement_comp

# Apply one-hot encoding to the same categorical features as the training data
encoded_features = ['ReferralSource', 'BookingChannel', 'AgeGroup', 'PackageType', 'Region', 'PromoCode']
X_comp = pd.get_dummies(X_comp, columns=encoded_features, dummy_na=False)

# Convert boolean columns (from get_dummies) to integers
for col in X_comp.select_dtypes(include='bool').columns:
    X_comp[col] = X_comp[col].astype(int)

# Drop any remaining categorical features that were not specified for encoding
X_comp = X_comp.drop(['BookingDate', 'Room'], axis=1, errors='ignore')

# Align columns - this is crucial to ensure both X and X_comp have the exact same columns
# after one-hot encoding, even if some categories are missing in one set.
# This gets the columns from the training set (X) and applies them to the test set (X_comp).
# Use X_columns_after_preprocessing as X is now a numpy array
missing_cols = set(X_columns_after_preprocessing) - set(X_comp.columns)
for c in missing_cols:
    X_comp[c] = 0
# Ensure the order of columns in the test set is the same as in the training set
X_comp = X_comp[X_columns_after_preprocessing]

# Code to convert all the column datatype to int.
X_comp = imp.transform(X_comp).astype(int)

# Use your model to make predictions
pred_comp = clf.predict(X_comp)

# Create a simple dataframe with two columns: the passenger ID (just the same as the test data) and our predictions
my_submission = pd.DataFrame({
    'GuestID': passengerIDs,
    'Churned': pred_comp})

# Let's take a peak at the results (as a sanity check)
display(my_submission.head(10))

# You could use any filename.
my_submission.to_csv('submission.csv', index=False)

# You can now download the 'submission.csv' from Colab/Kaggle (see menu on the left or right)

,GuestID,Churned
0,154038,1
1,620160,0
2,655103,0
3,126993,0
4,635228,0
5,844514,0
6,541503,1
7,787572,1
8,645651,0
9,849608,1
